In [0]:
import time
from datetime import datetime
import pyspark.sql.functions as F
from pyspark.sql.functions import broadcast

storage_account_name = "chungminhlamkhaiphadl"
storage_account_access_key = "your-key"
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_access_key)

start_time = datetime.now()

file_path = "abfss://reddit-data@chungminhlamkhaiphadl.dfs.core.windows.net/pushshift_*"

df_raw = spark.read.option("mergeSchema", "true").parquet(file_path)
df_submissions = df_raw.filter(F.col("name").startswith("t3_"))
# 3. BẢNG 1: BẢNG TRA CỨU (Lookup Table) - Chứa TẤT CẢ các bài viết
# Chỉ lấy cột 'name' (ID bài viết) và 'subreddit' (Nơi đăng)
df_lookup = df_submissions.select(
    F.col("name").alias("original_id"),
    F.col("subreddit").alias("source_subreddit")
).na.drop(subset=["original_id"]) # Bỏ qua các dòng lỗi không có ID

# 4. BẢNG 2: BẢNG CẦN TÌM (Edge Table) - Lọc cực mạnh chỉ lấy các bài LÀ CROSSPOST
df_crossposts = df_submissions.select(
    F.col("crosspost_parent"),
    F.col("subreddit").alias("target_subreddit")
).filter(
    F.col("crosspost_parent").isNotNull() & 
    (F.col("crosspost_parent") != "None") & 
    (F.col("crosspost_parent") != "")
)

# 5. PHÉP THUẬT BROADCAST HASH JOIN
# Dùng hàm broadcast() bọc bảng df_crossposts (bảng nhỏ) lại
df_joined = df_lookup.join(
    df_crossposts, 
    df_lookup["original_id"] == df_crossposts["crosspost_parent"], 
    how="inner"
)

# 6. GOM NHÓM ĐỂ TẠO TRỌNG SỐ (Cạnh của đồ thị)
# Đếm xem từ Subreddit A crosspost sang Subreddit B bao nhiêu lần
df_network = df_joined.groupBy("source_subreddit", "target_subreddit") \
                      .count() \
                      .withColumnRenamed("count", "weight")

display(df_network.orderBy(F.col("weight").desc()).limit(20))

df_network.coalesce(1).write.csv("abfss://reddit-data@chungminhlamkhaiphadl.dfs.core.windows.net/crosspost_result", header=True)

end_time = datetime.now()
print(start_time)
print(end_time)
print(f'Execution time: {end_time - start_time}')

source_subreddit,target_subreddit,weight
hackernews,patient_hackernews,54135
france,discussion_patiente,27956
mexico_politics,Mexico_News,26511
coins,MetalsOnReddit,21348
Silverbugs,MetalsOnReddit,18482
boardgames,PersonalizedGameRecs,18416
mexico,mejico,17308
Gold,MetalsOnReddit,16594
australia,Asked_Australia,14907
NoMansSkyTheGame,NMS_Discussions,14745


2026-04-09 08:05:05.179781
2026-04-09 09:22:00.389756
Execution time: 1:16:55.209975
